# Session 4 — GPU: Static vs Dynamic Computation Graph

## Background

Sessions 1–3 used a fully static forward pass: the computation graph is identical on every step, which is what XLA requires to fuse operations and avoid recompilation. Real research often involves models with data-dependent control flow — sparse attention patterns, conditional skip connections, variable-length routing. On GPU, PyTorch's eager execution handles these without penalty. On TPU, a Python `if` statement whose condition depends on a tensor value forces XLA to either retrace and recompile the graph or fall back to slower interpreted execution. This session isolates that cost directly, using the same Transformer block as the baseline to make the comparison exact.

## Goal

At the end of this session, participants will understand why dynamic control flow is the primary constraint on TPU suitability for certain research workloads, will have measured the throughput penalty of a data-dependent branch on each device, and will be able to refactor a branching `forward()` into a compilable equivalent — and know when refactoring is worth the effort vs when the GPU is simply the right tool.

## Question

How much throughput does a Python branch in `forward()` cost on a compiled (TPU) vs eager (GPU) device, and does a tensor-valued equivalent recover it? Do realistic dynamic patterns (padding masks, early exit, variable-length loops) follow the same pattern?

---

**Runtime:** GPU (NVIDIA L4 or equivalent)

After running, results are saved to `results/gpu_graph_variants.json` and loaded automatically by `03-analysis.ipynb`.

In [ ]:
import time, torch, torch.nn as nn

assert torch.cuda.is_available(), "No GPU found. Runtime → Change runtime type → GPU."

device = torch.device("cuda")
print(f"Device  : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")

Device  : NVIDIA L4
VRAM    : 23.7 GB
PyTorch : 2.10.0+cu128


In [ ]:
import sys; sys.path.insert(0, "..")
from transformer_block import BenchmarkTransformerBlock, D_MODEL, N_HEAD, DIM_FEEDFORWARD

# ── Session 4 config ──────────────────────────────────────────────────────────
BATCH_SIZE = 256
SEQ_LEN    = 128
N_STEPS, WARMUP = 50, 5

# ── Model variants ────────────────────────────────────────────────────────────

class StaticFF(nn.Module):
    """Baseline — identical to Sessions 1 and 2. Fully static graph."""
    def __init__(self):
        super().__init__()
        self.block = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)
    def forward(self, x):
        return self.block(x)


class DynamicFF(nn.Module):
    """Adds a Python if-else whose condition is a tensor value.
    On GPU: evaluated eagerly — zero penalty.
    On TPU: forces XLA to sync and potentially retrace the graph every step.
    """
    def __init__(self):
        super().__init__()
        self.block = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)
    def forward(self, x):
        out = self.block(x)
        if out.mean() > 0:   # Python branch on a tensor value
            return out
        else:
            return -out


class StaticEquivalentFF(nn.Module):
    """Replaces the Python branch with torch.where — a tensor op that stays in the XLA graph.
    Semantically equivalent to DynamicFF; XLA-compilable.
    """
    def __init__(self):
        super().__init__()
        self.block = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)
    def forward(self, x):
        out = self.block(x)
        return torch.where(out.mean() > 0, out, -out)  # condition stays in the graph


VARIANTS = {
    "StaticFF":           StaticFF,
    "DynamicFF":          DynamicFF,
    "StaticEquivalentFF": StaticEquivalentFF,
}

# ── Benchmark function ────────────────────────────────────────────────────────
def benchmark(model_class, device):
    model     = model_class().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()
    model.train()
    elapsed = 0.0
    for step in range(N_STEPS + WARMUP):
        x = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)
        y = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        if step >= WARMUP:
            elapsed += t1 - t0
    throughput = (N_STEPS * BATCH_SIZE) / elapsed
    latency_ms = (elapsed / N_STEPS) * 1000
    return throughput, latency_ms

print(f"Config  : batch={BATCH_SIZE}, seq_len={SEQ_LEN}, d_model={D_MODEL}")
print("Variants and benchmark function defined.")

Config  : batch=256, seq_len=128, d_model=512
Variants and benchmark function defined.


In [ ]:
results = {}

for name, cls in VARIANTS.items():
    print(f"  [GPU] {name:<22} ... ", end="", flush=True)
    tput, lat = benchmark(cls, device)
    results[name] = {"throughput": round(tput, 1), "latency_ms": round(lat, 2)}
    print(f"{tput:,.0f} samples/sec  ({lat:.1f} ms/step)")

print("\nDone.")

# Show relative slowdown vs StaticFF baseline
baseline = results["StaticFF"]["throughput"]
print("\n── Relative throughput (vs StaticFF) ──────────────")
for name, r in results.items():
    ratio = r["throughput"] / baseline
    print(f"  {name:<22}: {ratio:.3f}×")

  [GPU] StaticFF               ... 2,645 samples/sec  (96.8 ms/step)
  [GPU] DynamicFF              ... 2,590 samples/sec  (98.8 ms/step)
  [GPU] StaticEquivalentFF     ... 2,551 samples/sec  (100.4 ms/step)

Done.

── Relative throughput (vs StaticFF) ──────────────
  StaticFF              : 1.000×
  DynamicFF             : 0.979×
  StaticEquivalentFF    : 0.964×


---

## Realistic Scenarios  `# [PENDING RUN]`

The three abstract variants above (StaticFF / DynamicFF / StaticEquivalentFF) isolate the pure overhead of a data-dependent branch. The cells below benchmark three patterns that appear in real research code. Results are marked **[PENDING RUN]** — run them to populate the data.

### Scenario 1 — Padding mask (variable-length sequences)

In a real batch, sequences have different lengths and are padded to the maximum. A boolean mask is applied dynamically to the attention scores. On GPU (eager), masking is a standard tensor op. On TPU, if the mask shape or sparsity pattern changes step-to-step, XLA may need to retrace.

### Scenario 2 — Conditional early exit

A per-sample confidence score triggers early exit from the encoder stack when the score exceeds a threshold. The naive Python `if` implementation forces XLA to sync; the `torch.where` version keeps execution in the graph.

### Scenario 3 — Variable-length batch loop

Iterating over batch elements with per-sample variable depth (e.g., document QA with different numbers of passages). XLA is sensitive to the loop iteration count — changing it forces a retrace.

In [ ]:
# [PENDING RUN] Scenario 1 — Padding mask
# Run this cell to benchmark the padding-mask pattern on GPU.

import torch.nn.functional as F

class PaddingMaskFF(nn.Module):
    """Applies a boolean padding mask to attention scores.
    The mask varies per batch but its shape is constant — safe for XLA.
    Represents: NLP batch with variable-length sequences padded to SEQ_LEN.
    """
    def __init__(self):
        super().__init__()
        self.attn = nn.MultiheadAttention(D_MODEL, N_HEAD, batch_first=True)
        self.ff1  = nn.Linear(D_MODEL, DIM_FEEDFORWARD)
        self.ff2  = nn.Linear(DIM_FEEDFORWARD, D_MODEL)
        self.ln1  = nn.LayerNorm(D_MODEL)
        self.ln2  = nn.LayerNorm(D_MODEL)

    def forward(self, x, key_padding_mask):
        # key_padding_mask: (batch, seq_len) bool — True = position is padding
        attn_out, _ = self.attn(x, x, x, key_padding_mask=key_padding_mask)
        x = self.ln1(x + attn_out)
        ff_out = self.ff2(F.gelu(self.ff1(x)))
        return self.ln2(x + ff_out)


def benchmark_padding_mask(device, n_steps=N_STEPS, warmup=WARMUP):
    model     = PaddingMaskFF().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()
    model.train()
    elapsed = 0.0
    for step in range(n_steps + warmup):
        # Each step: random sequence lengths → different masks, same shape
        seq_lens = torch.randint(64, SEQ_LEN + 1, (BATCH_SIZE,))
        mask = torch.arange(SEQ_LEN).unsqueeze(0) >= seq_lens.unsqueeze(1)  # (B, S)
        x = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)
        y = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)
        mask = mask.to(device)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        optimizer.zero_grad()
        loss = criterion(model(x, mask), y)
        loss.backward()
        optimizer.step()
        torch.cuda.synchronize()
        if step >= warmup:
            elapsed += time.perf_counter() - t0
    return (n_steps * BATCH_SIZE) / elapsed, (elapsed / n_steps) * 1000


print("[GPU] Scenario 1 — Padding mask ... ", end="", flush=True)
tput, lat = benchmark_padding_mask(device)
results["PaddingMask"] = {"throughput": round(tput, 1), "latency_ms": round(lat, 2)}
print(f"{tput:,.0f} samples/sec  ({lat:.1f} ms/step)")

In [ ]:
# [PENDING RUN] Scenario 2 — Conditional early exit (Python if vs torch.where)

class EarlyExitDynamic(nn.Module):
    """Python if-branch on a per-sample confidence score.
    On GPU: eager, zero penalty. On TPU: forces XLA sync each step.
    """
    def __init__(self):
        super().__init__()
        self.block      = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)
        self.confidence = nn.Linear(D_MODEL, 1)
        self.threshold  = 0.5

    def forward(self, x):
        out   = self.block(x)
        score = torch.sigmoid(self.confidence(out[:, 0, :])).mean()  # scalar
        if score.item() > self.threshold:   # Python branch — forces XLA sync
            return out
        else:
            return self.block(out)           # second pass if not confident


class EarlyExitStatic(nn.Module):
    """torch.where replaces the Python branch — XLA-compilable."""
    def __init__(self):
        super().__init__()
        self.block      = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)
        self.confidence = nn.Linear(D_MODEL, 1)
        self.threshold  = 0.5

    def forward(self, x):
        out    = self.block(x)
        score  = torch.sigmoid(self.confidence(out[:, 0, :])).mean()
        out2   = self.block(out)             # always compute second pass
        return torch.where(score > self.threshold, out, out2)  # select in-graph


for name, cls in [("EarlyExitDynamic", EarlyExitDynamic), ("EarlyExitStatic", EarlyExitStatic)]:
    print(f"[GPU] Scenario 2 — {name} ... ", end="", flush=True)
    tput, lat = benchmark(cls, device)
    results[name] = {"throughput": round(tput, 1), "latency_ms": round(lat, 2)}
    print(f"{tput:,.0f} samples/sec  ({lat:.1f} ms/step)")

In [ ]:
# [PENDING RUN] Scenario 3 — Variable-length batch loop
# Iterates over batch elements with a Python for-loop of fixed length N_PASSES.
# On TPU: changing N_PASSES between steps forces recompilation.
# Here we keep N_PASSES fixed to show the XLA-friendly baseline,
# and contrast it with a version that changes the loop count each step.

N_PASSES_FIXED = 3

class FixedLoopFF(nn.Module):
    """Fixed number of passes — XLA traces a static loop, no recompilation."""
    def __init__(self):
        super().__init__()
        self.block = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)

    def forward(self, x):
        for _ in range(N_PASSES_FIXED):   # constant at trace time
            x = self.block(x)
        return x


class VariableLoopFF(nn.Module):
    """Loop count driven by a tensor value — forces XLA sync to evaluate."""
    def __init__(self):
        super().__init__()
        self.block = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD)

    def forward(self, x, n_passes):
        # n_passes is a Python int derived from a tensor value (syncs XLA on TPU)
        for _ in range(n_passes):
            x = self.block(x)
        return x


def benchmark_variable_loop(device, n_steps=N_STEPS, warmup=WARMUP):
    model     = VariableLoopFF().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()
    model.train()
    elapsed = 0.0
    for step in range(n_steps + warmup):
        # Alternate between 2 and 4 passes — changes XLA graph each step on TPU
        n_passes = 2 + (step % 3)
        x = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)
        y = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        optimizer.zero_grad()
        loss = criterion(model(x, n_passes), y)
        loss.backward()
        optimizer.step()
        torch.cuda.synchronize()
        if step >= warmup:
            elapsed += time.perf_counter() - t0
    return (n_steps * BATCH_SIZE) / elapsed, (elapsed / n_steps) * 1000


print("[GPU] Scenario 3 — FixedLoopFF     ... ", end="", flush=True)
tput, lat = benchmark(FixedLoopFF, device)
results["FixedLoopFF"] = {"throughput": round(tput, 1), "latency_ms": round(lat, 2)}
print(f"{tput:,.0f} samples/sec  ({lat:.1f} ms/step)")

print("[GPU] Scenario 3 — VariableLoopFF  ... ", end="", flush=True)
tput, lat = benchmark_variable_loop(device)
results["VariableLoopFF"] = {"throughput": round(tput, 1), "latency_ms": round(lat, 2)}
print(f"{tput:,.0f} samples/sec  ({lat:.1f} ms/step)")

In [ ]:
import json, os
from datetime import datetime, timezone

def save_results(hardware, results, device_name="", batch_size=None, seq_len=None, results_dir="results"):
    os.makedirs(results_dir, exist_ok=True)
    payload = {
        "hardware":    hardware,
        "device_name": device_name,
        "session":     "4",
        "batch_size":  batch_size,
        "seq_len":     seq_len,
        "timestamp":   datetime.now(timezone.utc).isoformat(),
        "results":     results,
    }
    path = os.path.join(results_dir, f"{hardware.lower()}_graph_variants.json")
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    return path

path = save_results(
    "GPU", results,
    device_name=torch.cuda.get_device_name(0),
    batch_size=BATCH_SIZE,
    seq_len=SEQ_LEN,
    results_dir="results",
)
print(f"Saved → {path}")

Saved → results/gpu_graph_variants.json
